# Multi-Use-Case Evaluation Metrics Notebook

This notebook evaluates the Reducto OracleDB Q&A layer across all four demos:

- financial
- healthcare
- insurance
- legal

Each evaluation case is run multiple times so latency and scoring metrics are less dependent on a single request. The notebook uses only the standard library plus the project code.

## Platform Context

![Executive thesis: Reducto AI + Oracle Database](assets/reducto_oracle_executive_thesis.svg)

![Capability map: Oracle Database approach vs standalone vector platforms](assets/reducto_oracle_platform_capability_map.svg)

![Operating model and risk controls](assets/reducto_oracle_operating_model_risk_controls.svg)


## Setup

In [1]:
import os
import re
import sys
import time
from pathlib import Path
from statistics import mean, median
from collections import defaultdict


def find_sdk_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "src" / "reducto" / "lib" / "oracledb").exists():
            return path
    raise RuntimeError("Could not locate the reducto-python-sdk repository root")


ROOT = find_sdk_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_env(path=ROOT / "examples" / "oracledb" / ".env"):
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[len("export ") :].strip()
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip("'\"")


load_env()

EVAL_REPEATS = int(os.getenv("EVAL_REPEATS", "3"))
EVAL_MODE = os.getenv("EVAL_MODE", "semantic").strip().lower()
SEARCH_LIMIT = int(os.getenv("EVAL_SEARCH_LIMIT", "5"))
EVIDENCE_LIMIT = int(os.getenv("EVAL_EVIDENCE_LIMIT", "3"))

print("Project root:", ROOT)
print("Oracle user:", os.getenv("ORACLE_USER"))
print("Oracle DSN:", os.getenv("ORACLE_DSN"))
print("Cohere key set:", bool(os.getenv("CO_API_KEY")))
print("Evaluation repeats:", EVAL_REPEATS)
print("Evaluation mode:", EVAL_MODE)

Oracle user: REDUCTO_RAG_APP
Oracle DSN: localhost:1521/FREEPDB1
Cohere key set: True
Evaluation repeats: 3
Evaluation mode: semantic


## Connect to Oracle

In [2]:
from reducto.lib.oracledb.qa import answer_from_search_results
from reducto.lib.oracledb.utils import read_lob
from reducto.lib.oracledb.config import connect_oracle, vector_dimensions_from_env
from reducto.lib.oracledb.models import SearchResult, SearchFilters
from reducto.lib.oracledb.oracle import OracleSchemaManager
from reducto.lib.oracledb.retrieval import OracleHybridRetriever
from reducto.lib.oracledb.embeddings import embedding_provider_from_env

connection = connect_oracle()
OracleSchemaManager(connection).create_schema(vector_dimensions=vector_dimensions_from_env())

embedding_provider = None
retriever = None
if EVAL_MODE in {"semantic", "hybrid"}:
    embedding_provider = embedding_provider_from_env(input_type="search_query")
    retriever = OracleHybridRetriever(connection, embedding_provider)

with connection.cursor() as cursor:
    cursor.execute(
        """
        select filing_type, company, fiscal_year, count(*)
        from documents
        group by filing_type, company, fiscal_year
        order by filing_type, company, fiscal_year
        """
    )
    inventory = cursor.fetchall()

for row in inventory:
    print(row)

('10-K', 'AAPL', 2023, 1)
('HEALTHCARE', 'MEDLINEPLUS', 2024, 1)
('INSURANCE', 'CMS', 2024, 1)
('LEGAL', 'SCOTUS', 2024, 1)


## Evaluation Set

The filters match the demo notebooks' document tags. If a document is missing, its cases are marked skipped instead of failing the whole benchmark.

In [3]:
def case(domain, case_id, question, filters, expected_phrases):
    return {
        "domain": domain,
        "id": case_id,
        "question": question,
        "filters": filters,
        "expected_phrases": expected_phrases,
    }


eval_set = [
    case(
        "financial",
        "aapl_distribution_channels_2023",
        "How did Apple distribute net sales through direct and indirect channels in 2023?",
        SearchFilters(company="AAPL", fiscal_year=2023, filing_type="10-K"),
        ["37%", "63%", "direct", "indirect"],
    ),
    case(
        "healthcare",
        "metformin_use",
        "What is metformin used to treat?",
        SearchFilters(company="MEDLINEPLUS", fiscal_year=2024, filing_type="HEALTHCARE"),
        ["type 2 diabetes", "blood sugar", "insulin"],
    ),
    case(
        "healthcare",
        "metformin_precautions",
        "What serious warning or precaution is mentioned for metformin?",
        SearchFilters(company="MEDLINEPLUS", fiscal_year=2024, filing_type="HEALTHCARE"),
        ["lactic acidosis", "kidney disease", "heart or liver disease"],
    ),
    case(
        "insurance",
        "sbc_cost_sharing",
        "What does the Summary of Benefits and Coverage help people understand?",
        SearchFilters(company="CMS", fiscal_year=2024, filing_type="INSURANCE"),
        ["share the cost", "covered health care services", "premium"],
    ),
    case(
        "insurance",
        "sbc_coverage_examples",
        "What cost sharing terms appear in the coverage examples?",
        SearchFilters(company="CMS", fiscal_year=2024, filing_type="INSURANCE"),
        ["deductibles", "copayments", "coinsurance", "excluded services"],
    ),
    case(
        "legal",
        "trump_v_anderson_holding",
        "What did the Supreme Court decide about Colorado enforcing Section 3 against federal candidates?",
        SearchFilters(company="SCOTUS", fiscal_year=2024, filing_type="LEGAL"),
        ["states have no power", "Congress", "Section 3", "federal officeholders"],
    ),
    case(
        "legal",
        "trump_v_anderson_reverse",
        "Why did the Court reverse the Colorado Supreme Court?",
        SearchFilters(company="SCOTUS", fiscal_year=2024, filing_type="LEGAL"),
        ["Congress", "rather than the States", "reverse", "federal officeholders"],
    ),
]

print("Evaluation cases:", len(eval_set))
for item in eval_set:
    print(item["domain"], item["id"], item["question"])

Evaluation cases: 7
financial aapl_distribution_channels_2023 How did Apple distribute net sales through direct and indirect channels in 2023?
healthcare metformin_use What is metformin used to treat?
healthcare metformin_precautions What serious warning or precaution is mentioned for metformin?
insurance sbc_cost_sharing What does the Summary of Benefits and Coverage help people understand?
insurance sbc_coverage_examples What cost sharing terms appear in the coverage examples?
legal trump_v_anderson_holding What did the Supreme Court decide about Colorado enforcing Section 3 against federal candidates?
legal trump_v_anderson_reverse Why did the Court reverse the Colorado Supreme Court?


## Metric Helpers

In [4]:
STOPWORDS = {
    "the",
    "and",
    "for",
    "that",
    "with",
    "what",
    "were",
    "was",
    "did",
    "does",
    "about",
    "from",
    "this",
    "into",
    "through",
    "against",
    "people",
    "court",
}

BOOL_METRICS = [
    "answer_contains_expected",
    "evidence_returned",
    "evidence_mentions_expected",
    "source_returned",
    "page_returned",
]


def contains_any(text, phrases):
    lower_text = str(text or "").lower()
    return any(phrase.lower() in lower_text for phrase in phrases)


def significant_terms(text):
    terms = []
    for term in re.findall(r"[a-zA-Z0-9]+", str(text).lower()):
        if len(term) >= 3 and term not in STOPWORDS:
            terms.append(term)
    return list(dict.fromkeys(terms))


def term_coverage(terms, text):
    if not terms:
        return 1.0
    lower_text = str(text or "").lower()
    return sum(1 for term in terms if term in lower_text) / len(terms)


def document_count(filters):
    clauses = []
    params = {}
    if filters.company:
        clauses.append("upper(company) = upper(:company)")
        params["company"] = filters.company
    if filters.fiscal_year is not None:
        clauses.append("fiscal_year = :fiscal_year")
        params["fiscal_year"] = filters.fiscal_year
    if filters.filing_type:
        clauses.append("upper(filing_type) = upper(:filing_type)")
        params["filing_type"] = filters.filing_type
    where_sql = " and ".join(clauses) if clauses else "1 = 1"
    with connection.cursor() as cursor:
        cursor.execute(f"select count(*) from documents where {where_sql}", params)
        return int(cursor.fetchone()[0])


def lexical_search(query, filters, limit=5):
    terms = significant_terms(query)[:8]
    clauses = []
    params = {}
    if filters.company:
        clauses.append("upper(d.company) = upper(:company)")
        params["company"] = filters.company
    if filters.fiscal_year is not None:
        clauses.append("d.fiscal_year = :fiscal_year")
        params["fiscal_year"] = filters.fiscal_year
    if filters.filing_type:
        clauses.append("upper(d.filing_type) = upper(:filing_type)")
        params["filing_type"] = filters.filing_type
    term_clauses = []
    for index, term in enumerate(terms):
        key = f"term_{index}"
        term_clauses.append(f"lower(c.content) like :{key}")
        params[key] = f"%{term}%"
    if term_clauses:
        clauses.append("(" + " or ".join(term_clauses) + ")")
    where_sql = " and ".join(clauses) if clauses else "1 = 1"
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            select d.document_id, c.chunk_id, c.content, d.company, d.fiscal_year,
                   d.filing_type, c.page_start, c.page_end, d.source_uri
            from document_chunks c
            join documents d on d.document_id = c.document_id
            where {where_sql}
            order by c.created_at desc
            fetch first {max(1, int(limit))} rows only
            """,
            params,
        )
        rows = cursor.fetchall()
    results = []
    for row in rows:
        content = str(read_lob(row[2]) or "")
        score = sum(1 for term in terms if term in content.lower())
        results.append(
            SearchResult(
                document_id=int(row[0]),
                chunk_id=int(row[1]),
                score=float(score),
                content=content,
                company=row[3],
                fiscal_year=int(row[4]) if row[4] is not None else None,
                filing_type=row[5],
                page_start=int(row[6]) if row[6] is not None else None,
                page_end=int(row[7]) if row[7] is not None else None,
                source_uri=str(row[8] or ""),
            )
        )
    return results


def search(case_item):
    if EVAL_MODE == "hybrid":
        return retriever.hybrid_search(case_item["question"], filters=case_item["filters"], limit=SEARCH_LIMIT)
    if EVAL_MODE == "lexical":
        return lexical_search(case_item["question"], case_item["filters"], limit=SEARCH_LIMIT)
    return retriever.semantic_search(case_item["question"], filters=case_item["filters"], limit=SEARCH_LIMIT)


def evaluate_case(case_item, repeat_index):
    if document_count(case_item["filters"]) == 0:
        return {
            "domain": case_item["domain"],
            "id": case_item["id"],
            "question": case_item["question"],
            "repeat": repeat_index,
            "status": "skipped_missing_document",
            "latency_ms": 0.0,
            "result_count": 0,
            "accuracy_score": 0,
            "top_search_score": 0.0,
            "answer": "",
            **{name: False for name in BOOL_METRICS},
            "question_coverage": 0.0,
            "answer_support": 0.0,
        }

    started = time.perf_counter()
    try:
        search_results = search(case_item)
        answer = answer_from_search_results(
            case_item["question"],
            search_results,
            evidence_limit=EVIDENCE_LIMIT,
        )
        latency_ms = round((time.perf_counter() - started) * 1000, 2)
        evidence_text = " ".join(item.text for item in answer.evidence)
        expected = case_item["expected_phrases"]
        answer_contains_expected = contains_any(answer.answer, expected)
        evidence_mentions_expected = contains_any(evidence_text, expected)
        evidence_returned = len(answer.evidence) > 0
        source_returned = any(item.source_uri for item in answer.evidence)
        page_returned = any(item.page_number is not None for item in answer.evidence)
        question_coverage = term_coverage(significant_terms(case_item["question"]), evidence_text)
        answer_support = term_coverage(significant_terms(answer.answer), evidence_text)
        scored = [
            evidence_returned,
            source_returned,
            page_returned,
            question_coverage >= 0.5,
            answer_support >= 0.65,
            answer_contains_expected,
            evidence_mentions_expected,
        ]
        return {
            "domain": case_item["domain"],
            "id": case_item["id"],
            "question": case_item["question"],
            "repeat": repeat_index,
            "status": "ok",
            "latency_ms": latency_ms,
            "result_count": len(search_results),
            "accuracy_score": round(100 * sum(scored) / len(scored)),
            "answer_contains_expected": answer_contains_expected,
            "evidence_returned": evidence_returned,
            "evidence_mentions_expected": evidence_mentions_expected,
            "source_returned": source_returned,
            "page_returned": page_returned,
            "question_coverage": round(question_coverage, 3),
            "answer_support": round(answer_support, 3),
            "top_search_score": round(max((result.score for result in search_results), default=0.0), 4),
            "answer": answer.answer,
        }
    except Exception as exc:
        return {
            "domain": case_item["domain"],
            "id": case_item["id"],
            "question": case_item["question"],
            "repeat": repeat_index,
            "status": f"error: {type(exc).__name__}: {exc}",
            "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            "result_count": 0,
            "accuracy_score": 0,
            "top_search_score": 0.0,
            "answer": "",
            **{name: False for name in BOOL_METRICS},
            "question_coverage": 0.0,
            "answer_support": 0.0,
        }

## Run Evaluation

In [5]:
results = []
for repeat_index in range(1, EVAL_REPEATS + 1):
    print(f"Run {repeat_index}/{EVAL_REPEATS}")
    for case_item in eval_set:
        row = evaluate_case(case_item, repeat_index)
        results.append(row)
        print(
            f"  {row['domain']:<10} {row['id']:<34} "
            f"status={row['status']} latency_ms={row['latency_ms']} "
            f"accuracy={row['accuracy_score']}"
        )

print("Total runs:", len(results))

Run 1/3
  financial  aapl_distribution_channels_2023    status=ok latency_ms=1382.86 accuracy=57
  healthcare metformin_use                      status=ok latency_ms=1190.44 accuracy=86
  healthcare metformin_precautions              status=ok latency_ms=1216.57 accuracy=86
  insurance  sbc_cost_sharing                   status=ok latency_ms=1255.94 accuracy=86
  insurance  sbc_coverage_examples              status=ok latency_ms=1199.38 accuracy=100
  legal      trump_v_anderson_holding           status=ok latency_ms=1294.77 accuracy=100
  legal      trump_v_anderson_reverse           status=ok latency_ms=1255.99 accuracy=86
Run 2/3
  financial  aapl_distribution_channels_2023    status=ok latency_ms=1328.33 accuracy=57
  healthcare metformin_use                      status=ok latency_ms=1157.42 accuracy=86
  healthcare metformin_precautions              status=ok latency_ms=1367.58 accuracy=86
  insurance  sbc_cost_sharing                   status=ok latency_ms=1184.06 accuracy=86
  i

## Per-Run Metrics

In [6]:
def format_table(rows, columns):
    if not rows:
        return "<no rows>"
    rendered_rows = []
    for row in rows:
        rendered_rows.append([str(row.get(column, "")) for column in columns])
    widths = [
        max(len(str(column)), *(len(row[index]) for row in rendered_rows)) for index, column in enumerate(columns)
    ]
    header = " | ".join(str(column).ljust(widths[index]) for index, column in enumerate(columns))
    divider = "-+-".join("-" * width for width in widths)
    body = "\n".join(
        " | ".join(row[index].ljust(widths[index]) for index in range(len(columns))) for row in rendered_rows
    )
    return f"{header}\n{divider}\n{body}"


per_run_columns = [
    "domain",
    "id",
    "repeat",
    "status",
    "latency_ms",
    "accuracy_score",
    "result_count",
    "top_search_score",
    "answer_contains_expected",
    "evidence_mentions_expected",
    "source_returned",
    "page_returned",
]
print(format_table(results, per_run_columns))

domain     | id                              | repeat | status | latency_ms | accuracy_score | result_count | top_search_score | answer_contains_expected | evidence_mentions_expected | source_returned | page_returned
-----------+---------------------------------+--------+--------+------------+----------------+--------------+------------------+--------------------------+----------------------------+-----------------+--------------
financial  | aapl_distribution_channels_2023 | 1      | ok     | 1382.86    | 57             | 1            | 0.0675           | False                    | False                      | True            | True         
healthcare | metformin_use                   | 1      | ok     | 1190.44    | 86             | 5            | 0.5663           | False                    | True                       | True            | True         
healthcare | metformin_precautions           | 1      | ok     | 1216.57    | 86             | 5            | 0.622            | Fal

## Aggregate Metrics

In [7]:
def percentile(values, pct):
    if not values:
        return 0.0
    ordered = sorted(values)
    index = min(len(ordered) - 1, max(0, round((pct / 100) * (len(ordered) - 1))))
    return ordered[index]


def summarize_rows(rows, label):
    successful = [row for row in rows if row["status"] == "ok"]
    latencies = [row["latency_ms"] for row in successful]
    summary = {
        "group": label,
        "runs": len(rows),
        "ok_runs": len(successful),
        "accuracy_avg": round(mean(row["accuracy_score"] for row in successful), 2) if successful else 0.0,
        "latency_avg_ms": round(mean(latencies), 2) if latencies else 0.0,
        "latency_median_ms": round(median(latencies), 2) if latencies else 0.0,
        "latency_p95_ms": round(percentile(latencies, 95), 2) if latencies else 0.0,
        "result_count_avg": round(mean(row["result_count"] for row in successful), 2) if successful else 0.0,
        "top_search_score_avg": round(mean(row["top_search_score"] for row in successful), 4) if successful else 0.0,
    }
    for metric in BOOL_METRICS:
        summary[f"{metric}_rate"] = round(sum(1 for row in successful if row[metric]) / max(1, len(successful)), 3)
    return summary


case_groups = defaultdict(list)
domain_groups = defaultdict(list)
for row in results:
    case_groups[row["id"]].append(row)
    domain_groups[row["domain"]].append(row)

case_summary = [summarize_rows(rows, case_id) for case_id, rows in case_groups.items()]
domain_summary = [summarize_rows(rows, domain) for domain, rows in domain_groups.items()]
overall_summary = summarize_rows(results, "overall")

summary_columns = [
    "group",
    "runs",
    "ok_runs",
    "accuracy_avg",
    "latency_avg_ms",
    "latency_median_ms",
    "latency_p95_ms",
    "result_count_avg",
    "top_search_score_avg",
    "answer_contains_expected_rate",
    "evidence_returned_rate",
    "evidence_mentions_expected_rate",
    "source_returned_rate",
    "page_returned_rate",
]

print("Per-case summary")
print(format_table(case_summary, summary_columns))
print("\nPer-domain summary")
print(format_table(domain_summary, summary_columns))
print("\nOverall summary")
print(format_table([overall_summary], summary_columns))

Per-case summary
group                           | runs | ok_runs | accuracy_avg | latency_avg_ms | latency_median_ms | latency_p95_ms | result_count_avg | top_search_score_avg | answer_contains_expected_rate | evidence_returned_rate | evidence_mentions_expected_rate | source_returned_rate | page_returned_rate
--------------------------------+------+---------+--------------+----------------+-------------------+----------------+------------------+----------------------+-------------------------------+------------------------+---------------------------------+----------------------+-------------------
aapl_distribution_channels_2023 | 3    | 3       | 57           | 1315.99        | 1328.33           | 1382.86        | 1                | 0.0675               | 0.0                           | 1.0                    | 0.0                             | 1.0                  | 1.0               
metformin_use                   | 3    | 3       | 86           | 1188.49        | 1190.44        

## Representative Answers

In [8]:
seen = set()
for row in results:
    key = row["id"]
    if key in seen or row["status"] != "ok":
        continue
    seen.add(key)
    print(f"[{row['domain']}] {row['id']}")
    print(f"Query: {row['question']}")
    print(f"Answer: {row['answer']}")
    print()

[financial] aapl_distribution_channels_2023
Query: How did Apple distribute net sales through direct and indirect channels in 2023?
Answer: The Company's total net sales were $383.3 billion and net income was $97.0 billion during 2023.

[healthcare] metformin_use
Query: What is metformin used to treat?
Answer: If you experience any of the following symptoms, stop taking metformin and call your doctor immediately: extreme tiredness, weakness, or discomfort; nausea; vomiting; stomach pain; decreased appetite; deep and rapid breathing or shortness of breath; dizziness; lightheadedness; fast or slow heartbeat; muscle pain; or feeling cold, especially in your han

[healthcare] metformin_precautions
Query: What serious warning or precaution is mentioned for metformin?
Answer: - lack of energy or weakness • - change in sense of taste • - headache • - flushing of the skin • - nail changes • - muscle pain • - rash# Metformin (cont.) ## Some side effects can be serious.

[insurance] sbc_cost_sha

In [9]:
connection.close()
print("Connection closed")

Connection closed
